# Single-strike KF — IV smile vs. risk-neutral density

**Goal:** find a snapshot where the **IV smile looks perfectly fine** (smooth, no
obvious kinks) but the **risk-neutral density goes negative somewhere** — i.e. a
hidden *butterfly arbitrage*.

Layout: top panel = OTM IV smile, bottom panel = density q(K) ∝ ∂²C/∂K²
(Breeden–Litzenberger). Butterfly arbitrage ⇔ q(K) < 0.

The KF is the per-strike (individual) Kalman filter — same `build_iv_surface`
machinery as `Kalman_filter_iv_surface_from_iv.ipynb`.

In [49]:
import os
import sys
from datetime import date

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import polars as pl
import polars.selectors as cs
from plotly.subplots import make_subplots
from scipy.interpolate import CubicSpline

project_root = os.path.abspath("/Users/elie/Documents/GitHub/quant_data/")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ["QUANTDATA_RAW_DATA"] = "!/quantdata/raw"
os.environ["QUANTDATA_TRAINING_DATA"] = "!/quantdata/training"
os.environ["QUANTDATA_PROCESSING_DATA"] = "!/quantdata/processing"
os.environ["QUANTDATA_CACHE_MODE"] = "on"
os.environ["QUANTDATA_TEST_MODE"] = "0"

import quantdata as _  # noqa: F401

sys.path.insert(
    0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "shared_functions")
)

import arbitrage
import black_scholes

import personal.shared_functions.kalman_filters as kalman_filters
import personal.shared_functions.utils as utils
from personal.shared_functions.utils import parse_option_ticker
from quantdata.massive.client import MassiveClient

## 1. Load data & build the single-strike KF IV surface

In [50]:
trade_date = date(2024, 1, 3)
client = MassiveClient()
df_spy = client.fetch_trades("SPY", trade_date)
df_tickers = pl.read_csv(f"../spy_options_tickers_{trade_date}.csv")

In [51]:
# Single-strike (individual) Kalman filter on IV — calls + puts in one batched call.
df_quote_result, df_trade_result = kalman_filters.build_iv_surface(
    df_tickers=df_tickers,
    trade_date=trade_date,
    df_spy=df_spy,
    min_trades=0,
    simple=False,
)

Skipping O:SPY240105C00388000: no matched trades
Skipping O:SPY240105C00424000: no matched trades
Skipping O:SPY240105C00425000: no matched trades
Skipping O:SPY240105C00441000: no matched trades
Skipping O:SPY240105C00443000: no matched trades
Skipping O:SPY240105C00453000: no matched trades
O:SPY240105C00469000: dropping 2 bad quotes (crossed×2 (first=2024-01-03 11:49:45.633413-05:00))
O:SPY240105C00472000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 15:51:02.275318-05:00))
O:SPY240105C00473000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 10:29:17.744517-05:00))
O:SPY240105C00474000: dropping 4 bad quotes (crossed×4 (first=2024-01-03 10:25:56.925706-05:00))
O:SPY240105C00475000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 14:02:06.496803-05:00))
O:SPY240105C00477000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 10:00:44.569057-05:00))
O:SPY240105C00481000: dropping 46 bad quotes (sub-tick bid×46 (first=2024-01-03 16:00:02.443177-05:00))
O:SPY240105C00482000

In [52]:
df_iv_surface_long = df_quote_result
df_iv_surface_wide = utils.synchronize_iv_surface(df_iv_surface_long)
df_filtered = df_iv_surface_wide.filter(
    pl.all_horizontal(pl.all().is_not_null()) & pl.all_horizontal(cs.float().is_not_nan())
)

strikes_C = sorted(df_iv_surface_long.filter(pl.col("is_call"))["strike"].unique().to_list())
strikes_P = sorted(df_iv_surface_long.filter(~pl.col("is_call"))["strike"].unique().to_list())
# Whole-dollar strikes only — drop the .5 strikes (sparse, and they hurt the smile graphics).
otm_strikes = [k for k in sorted(set(strikes_C) | set(strikes_P)) if float(k).is_integer()]

_, expiry, _ = parse_option_ticker(
    df_tickers.filter(pl.col("ticker").str.contains("C"))["ticker"][0]
)
print(f"{df_filtered.shape[0]:,} clean synchronized snapshots; expiry = {expiry}")

2,238,662 clean synchronized snapshots; expiry = 2024-01-05 16:00:00-04:56


## 2. Helpers — OTM smile, density, butterfly check

`otm_smile(row, spy_mid)` rebuilds the single OTM IV per strike exactly like the
source notebook (intersect call/put bid-ask spreads, fall back to the OTM leg).

`density_from_smile(...)` interpolates the IV smile with a cubic spline onto a
fine strike grid, prices forward calls via Black, and takes the second derivative
→ risk-neutral density (undiscounted; only the **sign** matters for arbitrage).

`discrete_butterfly(...)` is the model-free arb on the *actual* strikes: the
second difference of forward call prices must be ≥ 0. This is the real,
interpolation-independent butterfly signal.

In [53]:
has_C_set = {k for k in otm_strikes if f"kalman_iv_{int(k)}_C" in df_filtered.columns}
has_P_set = {k for k in otm_strikes if f"kalman_iv_{int(k)}_P" in df_filtered.columns}

# Smile source: "otm" blends calls+puts (intersect spreads, OTM leg); "puts" / "calls"
# build the smile from that option type only.
SMILE_SOURCE = "puts"


def otm_smile(row, spy_mid):
    """Return (strikes, ivs) of the IV smile at one snapshot row.

    SMILE_SOURCE controls the construction:
      "puts"  -> put  KF mid IV at every strike that has a put
      "calls" -> call KF mid IV at every strike that has a call
      "otm"   -> OTM-blended (intersect call/put spreads, fall back to OTM leg)
    """
    ks, ivs = [], []
    for k in otm_strikes:
        K = int(k)
        has_C, has_P = k in has_C_set, k in has_P_set
        if SMILE_SOURCE == "puts":
            iv = row[f"kalman_iv_{K}_P"][0] if has_P else None
        elif SMILE_SOURCE == "calls":
            iv = row[f"kalman_iv_{K}_C"][0] if has_C else None
        elif has_C and has_P:
            bid_C, ask_C = row[f"bid_iv_{K}_C"][0], row[f"ask_iv_{K}_C"][0]
            bid_P, ask_P = row[f"bid_iv_{K}_P"][0], row[f"ask_iv_{K}_P"][0]
            if None in (bid_C, ask_C, bid_P, ask_P):
                continue
            i_bid, i_ask = max(bid_C, bid_P), min(ask_C, ask_P)
            if i_bid <= i_ask:
                iv = (i_bid + i_ask) / 2
            else:
                iv = row[f"kalman_iv_{K}_C"][0] if k > spy_mid else row[f"kalman_iv_{K}_P"][0]
        elif has_C:
            iv = row[f"kalman_iv_{K}_C"][0]
        elif has_P:
            iv = row[f"kalman_iv_{K}_P"][0]
        else:
            continue
        if iv is None:
            continue
        ks.append(float(k))
        ivs.append(float(iv))
    return np.array(ks), np.array(ivs)


def forward_calls(K, F, T, iv):
    """Undiscounted (forward-measure) Black call prices: c = F N(d1) - K N(d2)."""
    return np.asarray(black_scholes.bs_call_price(F, jnp.asarray(K), T, 0.0, jnp.asarray(iv)))


def density_from_smile(strikes, ivs, F, T, n_grid=400):
    """Breeden-Litzenberger RND on a fine grid via cubic-spline-interpolated IV."""
    cs_iv = CubicSpline(strikes, ivs)
    Kg = np.linspace(strikes.min(), strikes.max(), n_grid)
    iv_g = cs_iv(Kg)
    c = forward_calls(Kg, F, T, iv_g)
    q = np.gradient(np.gradient(c, Kg), Kg)  # d2C/dK2
    return Kg, iv_g, q


def discrete_butterfly(strikes, ivs, F, T):
    """Model-free butterfly on actual strikes — spacing-aware (handles mixed $1/$5).

    Convexity of C in K <=> no butterfly arb. With unequal strike spacing the test is
    that the call-price slope is non-decreasing: fly = slope_right - slope_left >= 0.
    Returns (strikes_mid, fly) where fly < 0 flags a butterfly arbitrage at that strike.
    """
    c = forward_calls(strikes, F, T, ivs)
    slope = np.diff(c) / np.diff(strikes)
    fly = slope[1:] - slope[:-1]
    return strikes[1:-1], fly

## 3. Scan every snapshot for "smooth smile, negative density"

For each snapshot we compute the discrete butterfly on the real strikes. A
snapshot is *arbitrageable* if any second-difference is negative. Among those we
rank by how **smooth / innocent** the smile looks — measured by the largest
absolute third difference of the IV (a kinky smile screams "arb", a smooth one
hides it). The lowest score = the most deceptive example.

In [54]:
def tte_at(ts):
    return float((expiry - ts).total_seconds()) / (365.25 * 24 * 3600)


def smile_roughness(ivs):
    """Max |3rd difference| of IV — proxy for how 'kinky' the smile looks to the eye."""
    if len(ivs) < 4:
        return np.inf
    d3 = np.diff(ivs, n=3)
    return float(np.max(np.abs(d3)))


# Subsample for speed; tighten the step for a finer scan.
STEP = 50
spy_sorted = df_spy.sort("ts_ns").select("ts_ns", "price")
sample = df_filtered[::STEP]
spy_mids = (
    sample.select("ts_ns").join_asof(spy_sorted, on="ts_ns", strategy="backward")["price"].to_list()
)

candidates = []
for j in range(sample.shape[0]):
    row = sample[j]
    ts = row["ts_ns"][0]
    spy_mid = spy_mids[j]
    if spy_mid is None:
        continue
    F = row["forward"][0] if "forward" in row.columns and row["forward"][0] is not None else spy_mid
    T = tte_at(ts)
    ks, ivs = otm_smile(row, spy_mid)
    if len(ks) < 5:
        continue
    _, fly = discrete_butterfly(ks, ivs, F, T)
    min_fly = float(fly.min())
    if min_fly < 0:  # butterfly arbitrage present
        candidates.append((smile_roughness(ivs), min_fly, j, ts, F, spy_mid))

candidates.sort(key=lambda c: c[0])  # smoothest-looking first
print(f"{len(candidates)} arbitrageable snapshots out of {sample.shape[0]} scanned")
for rough, min_fly, j, ts, F, spy_mid in candidates[:10]:
    print(f"  roughness={rough:.5f}  min_fly={min_fly:+.4f}  sample_idx={j}  ts={ts}")

44774 arbitrageable snapshots out of 44774 scanned
  roughness=0.02180  min_fly=-0.0463  sample_idx=1946  ts=2024-01-03 11:01:19.042631-05:00
  roughness=0.02180  min_fly=-0.0463  sample_idx=1947  ts=2024-01-03 11:01:19.603599-05:00
  roughness=0.02293  min_fly=-0.0531  sample_idx=1948  ts=2024-01-03 11:01:19.663934-05:00
  roughness=0.02327  min_fly=-0.0505  sample_idx=21297  ts=2024-01-03 14:02:18.507917-05:00
  roughness=0.02327  min_fly=-0.0504  sample_idx=21298  ts=2024-01-03 14:02:18.566305-05:00
  roughness=0.02332  min_fly=-0.0617  sample_idx=21299  ts=2024-01-03 14:02:18.742786-05:00
  roughness=0.02366  min_fly=-0.0645  sample_idx=21294  ts=2024-01-03 14:02:17.927293-05:00
  roughness=0.02397  min_fly=-0.0667  sample_idx=21291  ts=2024-01-03 14:02:17.301938-05:00
  roughness=0.02420  min_fly=-0.0682  sample_idx=21295  ts=2024-01-03 14:02:18.206651-05:00
  roughness=0.02420  min_fly=-0.0682  sample_idx=21296  ts=2024-01-03 14:02:18.438375-05:00


## 4. Plot the most deceptive example — smooth smile, hidden butterfly

In [63]:
# Pick the smoothest-looking arbitrageable snapshot (rank 0). Change to scrub.
RANK = 3
_, _, j, ts, F, spy_mid = candidates[RANK]
row = sample[j]
T = tte_at(ts)

ks, ivs = otm_smile(row, spy_mid)
Kg, iv_g, q = density_from_smile(ks, ivs, F, T)
k_mid, fly = discrete_butterfly(ks, ivs, F, T)

# Gatheral-Jacquier g(k) indicator for cross-check (log-moneyness space).
g_res = arbitrage.check_butterfly(k=np.log(ks / F), iv=ivs, T=T, plot=False)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.55, 0.45],
    subplot_titles=("OTM IV smile (single-strike KF mid)", "Risk-neutral density  q(K) ∝ ∂²C/∂K²"),
)

# --- Top: IV smile (linear interpolation between strikes, like a regular plot) ---
fig.add_trace(
    go.Scatter(
        x=ks,
        y=ivs,
        mode="lines+markers",
        name="KF OTM IV",
        line=dict(color="royalblue"),
        marker=dict(color="black", size=6),
    ),
    row=1,
    col=1,
)
fig.add_vline(x=F, line_dash="dash", line_color="gray", row=1, col=1, annotation_text=f"F={F:.2f}")

# --- Bottom: density, negative region shaded red ---
fig.add_trace(
    go.Scatter(x=Kg, y=q, mode="lines", name="density q(K)", line=dict(color="seagreen")),
    row=2,
    col=1,
)
q_neg = np.where(q < 0, q, np.nan)
fig.add_trace(
    go.Scatter(
        x=Kg, y=q_neg, mode="lines", name="butterfly arb (q<0)", line=dict(color="red", width=3)
    ),
    row=2,
    col=1,
)
fig.add_hline(y=0, line_dash="dot", line_color="gray", row=2, col=1)
# discrete butterfly violations on the real strikes
viol = k_mid[fly < 0]
for kv in viol:
    fig.add_vline(x=kv, line_color="red", line_width=1, opacity=0.25, row=2, col=1)

fig.update_layout(
    height=720,
    title=f"Hidden butterfly arbitrage — {ts}  |  min discrete fly = {fly.min():+.4f}  "
    f"|  g(k) min = {g_res['min_g']:+.4f}",
)
fig.update_xaxes(title_text="Strike", row=2, col=1)
fig.update_yaxes(title_text="Implied vol", row=1, col=1)
fig.update_yaxes(title_text="density", row=2, col=1)
fig.show()

print(f"Strikes with butterfly arb (q<0): {viol.tolist()}")
print(f"Smile looks arb-free to the eye? roughness={smile_roughness(ivs):.5f}")

Strikes with butterfly arb (q<0): [437.0, 441.0, 445.0, 449.0, 453.0, 455.0, 457.0, 460.0, 478.0, 480.0, 481.0, 483.0, 485.0, 493.0]
Smile looks arb-free to the eye? roughness=0.02327


### Notes / knobs
- **No candidates?** Lower `STEP` (finer scan), or relax the smoothness ranking.
- The **discrete** butterfly (`discrete_butterfly`) is the real arb on traded
  strikes; the spline density is for visualization and can add its own wiggles.
  Trust `min_fly < 0` for the actual signal.
- Scrub `RANK` to walk from smoothest→kinkiest examples.
- To inspect a specific timestamp instead of scanning, set `j` to a row of
  `sample` (or filter `df_filtered` directly) and re-run the plot cell.